In [1]:
#Very first step is to import these modules and libraries into the satprocessing kernel
import numpy as np
import dask as ds
import earthaccess as ea
import xarray as xr
import sys
import matplotlib.pyplot as plt
import requests
import pandas as pd
import PyCO2SYS as pyco2
print("module import success")

import sys
sys.path.append(r'C:\\Users\\edmund.gable\\Documents\\Hollings_2026\\RESOURCES\\python\\utilities\\bootstrap\\')
print("path import success")

import pprint
from environment import bootstrap_environment  
pprint.pprint(bootstrap_environment(preferred="edmund_gable"))
print("success")

from utilities.bootstrap.environment import bootstrap_environment
env = bootstrap_environment(preferred="edmund.gable")
env
print("utilities import success")

#Before this next step, install the library 'resource': conda install -c conda-forge resource
from utilities import get_dates, get_source_file_dates, find_text
find_text("stats")

from utilities import get_prod_files 
get_prod_files("DO", period="DOY")
#"DOY" is climatological 'day of year
#'W' is climatological weekly files

from pathlib import Path

root = Path(r"C:\Users\edmund.gable\Documents\Hollings_2026\GitHub\MOM6-COBALT_2026")
raw_data = Path(r"C:\Users\edmund.gable\Documents\Hollings_2026\GitHub\MOM6-COBALT_2026\data\Raw_Data")



c:\Users\edmund.gable\.conda\envs\satprocessing\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


module import success
path import success
⚠ Unrecognized preferred label 'edmund_gable'. Valid options: ['NECMAC04363461.local', 'nefscsatdata', 'guihyde', 'Mac.localdomain', 'gdavis', 'egable']
{'dataset_path': WindowsPath('C:/Users/edmund.gable/Documents/Hollings_2026/DATASETS'),
 'hostname': 'NECL04363598',
 'lookuptable_path': WindowsPath('C:/Users/edmund.gable/Documents/Hollings_2026/RESOURCES/workflow_resources/lookuptables'),
 'metadata_path': WindowsPath('C:/Users/edmund.gable/Documents/Hollings_2026/RESOURCES/workflow_resources/metadata'),
 'project_root': WindowsPath('C:/Users/edmund.gable/Documents/Hollings_2026/RESOURCES/python'),
 'python_path': WindowsPath('C:/Users/edmund.gable/Documents/Hollings_2026/RESOURCES/python'),
 'satlogs_path': WindowsPath('C:/Users/edmund.gable/Documents/Hollings_2026/RESOURCES/logs/satprocessing'),
 'timestamp': datetime.datetime(2026, 6, 10, 11, 27, 38, 427300),
 'utilities_path': WindowsPath('C:/Users/edmund.gable/Documents/Hollings_2026/RE

Steps for opening Glider Data or Buoy data
- Download data file into github folder 'data'
- In VScode pull datset from data folder into OA_data notebook
- identify structure of file (printout variable names and dimensions (lat lon depth time))
- identify variable names, units (aragonite, pH, carbon concentration, anything else) with index
- Determine criteria of values and variables
    - Time dimension of dates but want a date range
    - ex: January 24 to February 1st
        - specify dates matching time format in datafile
        - or make a broader description if they just give a month (01-31 of month)
    - Space dimension - nail down where it occured with coordinates (lat, long) (*N *E/W) but identify specific region and if its nearshore, offshore
        - with moving data: take all data within a 'box' of coordinates (minimum and maximum of latitude and longitude is bounding box)
        - With buoy data identify specific larger region around fixed point with a centered bounding box (+- 10km for lat and lon)
    - If data does not have saturated aragonite, create function for sat aragonite from pH and CO2 ppm
        - def sat_arag(ph, CO2)
    - For fixed point: Identify mininum saturated aragonite within that date range and identify as the event value
    - For a moving point: identify average of each day within box, then find minimum of all daily averages and identify as event value
    - Identify threshold for an event and create function 
        - If sat_arag < Longfin_threshold:
            - return sat_arag
    - sat_arag>event_data
    - make replicate file with dictionary that can receive all of the sat_arag data over time and space
        - ([start date],[end date], Box:[min longitude][max longitude][min lattitude][max lattitude], [min sat_arag],[below threshold?],[approaching threshold?])

- by saving this data in a table with a known format and NAMED VARIABLES (not hardcoded), you can read in MOM6 data and compare it with a function that runs repeatedly for all events
- output a text file/dictionary with keys and values
- Fill out sheet with dictionary values


Basically same for DO but switch out sat_aragewith mg/L O2, threshold is 2.31+-0.1



Functions List:

In [2]:

#Step 2: Define these functions
#Function to return the spatial extents for each vessel cruise or glider deployment by day, which will be used to create bounding boxes for the model grid
def get_extent(df, time_column, lon_column, lat_column, depth_column):
    # 1. Convert to datetime and strip hours/minutes/seconds (normalizes to midnight)
    daily_time = pd.to_datetime(df[time_column]).dt.normalize()
    
    # 2. Group by the new daily time
    lon_max = df.groupby(daily_time)[lon_column].max()
    lon_min = df.groupby(daily_time)[lon_column].min()
    lat_max = df.groupby(daily_time)[lat_column].max()
    lat_min = df.groupby(daily_time)[lat_column].min()
    depth_max = df.groupby(daily_time)[depth_column].max()
    depth_min = df.groupby(daily_time)[depth_column].min()
    # 3. Create the dataframe
    extent_df = pd.DataFrame({
        "lon_max": lon_max,
        "lon_min": lon_min,
        "lat_max": lat_max,
        "lat_min": lat_min,
        "depth_max": depth_max,
        "depth_min": depth_min
    })
    
    # 4. Reset index so 'time_column' becomes a regular column again instead of the index
    return extent_df.reset_index()
#vessel_extent = get_extent(df, "deployment", "lon", "lat", 'depth')
#print(vessel_extent)
#vessel_extent.to_excel(root / "data" / "Processed_Data" / "vessel__based_bottom_OA_extent_2007_2024.xlsx", index=False)

#Function to qualify the omega aragnoite saturation states of each data point in a file
def classify_arag(df, aragonite_column):
    classified_df=df.copy()
    classified_df.loc[classified_df[aragonite_column]>3.0, 'aragonite_status']='Optimal'
    classified_df.loc[classified_df[aragonite_column]>1.4, 'aragonite_status']='Saturated'
    classified_df.loc[classified_df[aragonite_column]<1.4, 'aragonite_status']='Suboptimal'
    classified_df.loc[classified_df[aragonite_column]<1.19, 'aragonite_status']='Atlantic Cod threshold surpassed'
    classified_df.loc[classified_df[aragonite_column]<1.1, 'aragonite_status']='Atlantic Sea Scallop threshold surpassed'
    classified_df.loc[classified_df[aragonite_column]<1, 'aragonite_status']='Critical threshold surpassed'
    classified_df=classified_df.reset_index()
    return classified_df

#Function to export the classified aragonite as a csv
def export_classified_arag(df, as_file_name):
    df_classified=classify_arag(df, "omega_bottom")
    df_classified
    df_classified.to_csv(root/"data"/"Processed_Data"/ as_file_name, index=False)

#Function that calculates the daily averages of saturated aragonite

def calculate_daily_averages(df, name_col, datetime_col, target_col):
    df_working = df.copy()
    
    # 1. Extract the date
    df_working['time'] = pd.to_datetime(df_working[datetime_col]).dt.date

    df_working['deployment'] = df_working[name_col]
    
    # 2. Ensure the raw aragonite column is numeric
    df_working[target_col] = pd.to_numeric(df_working[target_col], errors='coerce')
    
    # 3. Group by vessel and time, then calculate the mean of the raw numbers
    # FIX: Group by 'deployment' instead of 'datetime_col'
    daily_avg_df = df_working.groupby(['deployment', 'time'])[target_col].mean().reset_index()
    
    # 4. Rename the column so we know it is a daily average
    avg_col_name = f'{target_col}_daily_avg'
    daily_avg_df.rename(columns={target_col: avg_col_name}, inplace=True)
    
    return daily_avg_df

#Function which will return the final data file with spatial extent of the glider/vessel deployment and averaged+classified daily values of saturated aragonite
import pandas as pd

def combine_df(df1, df2, join_key1):
    # Make copies so we don't accidentally alter the original dataframes
    df1_working = df1.copy()
    df2_working = df2.copy()
    
    # Force the join key in both dataframes to be a standard Pandas datetime object
    df1_working[join_key1] = pd.to_datetime(df1_working[join_key1])
    df2_working[join_key1] = pd.to_datetime(df2_working[join_key1])
    
    # Now that the data types perfectly match, perform the merge
    combined_df = pd.merge(df1_working, df2_working, on=join_key1)
    
    return combined_df

#Function that checks the names of columns in a dataframe
def check_columns(df):
    df.reset_index()
    print(df.columns.to_list())

In [ ]:
#Step 3: Open datset and convert to a dataframe
dsg=xr.open_dataset(root/ "data" / "Raw_Data" / "Gliders" / "glider_based_bottom_OA_data_2019_2025.nc")
dfg=dsg.to_dataframe()
#Check the names of the columns for function inputs
print(dfg.columns.tolist())


['deployment', 'lat', 'lon', 'depth', 'bottom_depth', 'pH_bottom', 'omega_bottom', 'temperature_bottom']


In [ ]:
# 4. Start fresh from the original xarray dataset and reset the index immediately
dfg = dsg.to_dataframe()

# Print to verify your columns are clean
print("Original Columns:", dfg.columns.tolist())

# 5. Classify aragonite
dfgc = classify_arag(dfg, 'omega_bottom')
print("Classify Columns:", dfgc.columns.tolist())


Original Columns: ['deployment', 'lat', 'lon', 'depth', 'bottom_depth', 'pH_bottom', 'omega_bottom', 'temperature_bottom']
Classify Columns: ['time', 'deployment', 'lat', 'lon', 'depth', 'bottom_depth', 'pH_bottom', 'omega_bottom', 'temperature_bottom', 'aragonite_status']


,time,deployment,lat,lon,depth,bottom_depth,pH_bottom,omega_bottom,temperature_bottom,aragonite_status
0,2019-08-05 00:01:15.711111168,ru30-20190717T1812,39.809955,-73.212830,39.491982,43.0,7.8281,1.08,7.950000,Atlantic Sea Scallop threshold surpassed
1,2019-08-05 00:05:55.698529024,ru30-20190717T1812,39.810494,-73.214054,39.546547,43.0,7.8257,1.08,7.950000,Atlantic Sea Scallop threshold surpassed
2,2019-08-05 00:10:47.911289856,ru30-20190717T1812,39.811057,-73.215331,41.148884,43.0,7.8294,1.09,7.960000,Atlantic Sea Scallop threshold surpassed
3,2019-08-05 00:15:05.200000000,ru30-20190717T1812,39.811553,-73.216456,41.044708,43.0,7.8278,1.08,7.960000,Atlantic Sea Scallop threshold surpassed
4,2019-08-05 00:19:32.785124096,ru30-20190717T1812,39.812068,-73.217626,42.493248,43.0,7.8300,1.09,7.970000,Atlantic Sea Scallop threshold surpassed
...,...,...,...,...,...,...,...,...,...,...
134051,2025-08-11 14:30:33.000000000,ru39-20250716T1542,39.305976,-74.335720,12.836317,15.0,8.0142,2.69,23.660000,Saturated
134052,2025-08-11 14:45:56.144999936,ru39-20250716T1542,39.305388,-74.335010,12.325262,15.0,8.0202,2.72,23.670000,Saturated
134053,2025-08-11 14:46:08.020999936,ru39-20250716T1542,39.305386,-74.335017,12.196258,15.0,8.0215,2.72,23.670000,Saturated
134054,2025-08-11 14:54:37.000000000,ru39-20250716T1542,39.305121,-74.334853,12.141679,15.0,8.0223,2.73,23.690001,Saturated


In [5]:

#make sure to always double check column names before running functions
print(dfg.columns.tolist())
#When performing the get_extent() function it is important to first reset the dataframe index becasue pandas often ignores time columns as an index
dfg=dfg.reset_index()

dfgc=classify_arag(dfg, 'omega_bottom')
print(dfgc.columns.tolist())
dfgc

['deployment', 'lat', 'lon', 'depth', 'bottom_depth', 'pH_bottom', 'omega_bottom', 'temperature_bottom']
['index', 'time', 'deployment', 'lat', 'lon', 'depth', 'bottom_depth', 'pH_bottom', 'omega_bottom', 'temperature_bottom', 'aragonite_status']


,index,time,deployment,lat,lon,depth,bottom_depth,pH_bottom,omega_bottom,temperature_bottom,aragonite_status
0,0,2019-08-05 00:01:15.711111168,ru30-20190717T1812,39.809955,-73.212830,39.491982,43.0,7.8281,1.08,7.950000,Atlantic Sea Scallop threshold surpassed
1,1,2019-08-05 00:05:55.698529024,ru30-20190717T1812,39.810494,-73.214054,39.546547,43.0,7.8257,1.08,7.950000,Atlantic Sea Scallop threshold surpassed
2,2,2019-08-05 00:10:47.911289856,ru30-20190717T1812,39.811057,-73.215331,41.148884,43.0,7.8294,1.09,7.960000,Atlantic Sea Scallop threshold surpassed
3,3,2019-08-05 00:15:05.200000000,ru30-20190717T1812,39.811553,-73.216456,41.044708,43.0,7.8278,1.08,7.960000,Atlantic Sea Scallop threshold surpassed
4,4,2019-08-05 00:19:32.785124096,ru30-20190717T1812,39.812068,-73.217626,42.493248,43.0,7.8300,1.09,7.970000,Atlantic Sea Scallop threshold surpassed
...,...,...,...,...,...,...,...,...,...,...,...
134051,134051,2025-08-11 14:30:33.000000000,ru39-20250716T1542,39.305976,-74.335720,12.836317,15.0,8.0142,2.69,23.660000,Saturated
134052,134052,2025-08-11 14:45:56.144999936,ru39-20250716T1542,39.305388,-74.335010,12.325262,15.0,8.0202,2.72,23.670000,Saturated
134053,134053,2025-08-11 14:46:08.020999936,ru39-20250716T1542,39.305386,-74.335017,12.196258,15.0,8.0215,2.72,23.670000,Saturated
134054,134054,2025-08-11 14:54:37.000000000,ru39-20250716T1542,39.305121,-74.334853,12.141679,15.0,8.0223,2.73,23.690001,Saturated


In [ ]:

#dfgd is "dataframe glider daily", dfgdc is "dataframe glider daily classified"
dfgd=calculate_daily_averages(dfgc, "deployment", "time", "omega_bottom")


#check the new column names to perform a classify_arag() function on the averaged dataframe
check_columns(dfgd)
#6. classify your aragonite state from the daily averages
dfgdc=classify_arag(dfgd, 'omega_bottom_daily_avg')
print(dfgdc)

dfgd


['deployment', 'time', 'omega_bottom_daily_avg']
      index            deployment        time  omega_bottom_daily_avg  \
0         0   SBU01-20210226T1910  2021-02-26                1.760714   
1         1   SBU01-20210226T1910  2021-02-27                1.734255   
2         2   SBU01-20210226T1910  2021-02-28                1.778706   
3         3   SBU01-20210226T1910  2021-03-01                1.737808   
4         4   SBU01-20210226T1910  2021-03-02                1.910167   
...     ...                   ...         ...                     ...   
1123   1123  um_242-20231109T1526  2023-12-04                1.361463   
1124   1124  um_242-20231109T1526  2023-12-05                1.405556   
1125   1125  um_242-20231109T1526  2023-12-06                1.380746   
1126   1126  um_242-20231109T1526  2023-12-07                1.391720   
1127   1127  um_242-20231109T1526  2023-12-08                1.362308   

     aragonite_status  
0           Saturated  
1           Saturated  
2 

In [7]:
# 7. Get extent (Do NOT run dfg.reset_index() again here!)
print(dfg.columns.to_list())
dfge = get_extent(dfg, 'time', 'lon', 'lat', 'depth')
print("Extent Columns:", dfge.columns.tolist())
dfge=dfge.reset_index()
print(dfge)
print(dfgdc)

['time', 'deployment', 'lat', 'lon', 'depth', 'bottom_depth', 'pH_bottom', 'omega_bottom', 'temperature_bottom']
Extent Columns: ['time', 'lon_max', 'lon_min', 'lat_max', 'lat_min', 'depth_max', 'depth_min']
     index       time    lon_max    lon_min    lat_max    lat_min  depth_max  \
0        0 2019-08-05 -73.212830 -73.399686  39.819206  39.795079  43.758213   
1        1 2019-08-06 -73.399621 -73.556451  39.810909  39.792547  37.314186   
2        2 2019-08-07 -73.557035 -73.682571  39.851079  39.794003  36.570053   
3        3 2019-08-08 -73.682855 -73.769853  39.993008  39.837254  34.997208   
4        4 2019-08-09 -73.770255 -73.795513  40.146323  39.993122  33.935154   
..     ...        ...        ...        ...        ...        ...        ...   
824    824 2025-08-07 -74.049002 -74.234219  39.057192  38.958827  40.025871   
825    825 2025-08-08 -74.237490 -74.361961  39.157003  39.042188  28.440836   
826    826 2025-08-09 -74.303621 -74.363445  39.264279  39.159145  20.37

In [8]:

# Step 8: Combine with your spatial bounds (min/max lat and lon) according to the time column
df_final = combine_df(dfgdc, dfge, 'time')
print(df_final)
# Step 9: Export it 
export_path = root / "data" / "Processed_Data" / "glider_based_bottom_OA_extent_average_2019_2025.csv"
#df_final.to_csv(export_path, date_format='%Y-%m-%d %H:%M:%S', index=False)

      index_x            deployment       time  omega_bottom_daily_avg  \
0           0   SBU01-20210226T1910 2021-02-26                1.760714   
1           1   SBU01-20210226T1910 2021-02-27                1.734255   
2           2   SBU01-20210226T1910 2021-02-28                1.778706   
3           3   SBU01-20210226T1910 2021-03-01                1.737808   
4           4   SBU01-20210226T1910 2021-03-02                1.910167   
...       ...                   ...        ...                     ...   
1123     1123  um_242-20231109T1526 2023-12-04                1.361463   
1124     1124  um_242-20231109T1526 2023-12-05                1.405556   
1125     1125  um_242-20231109T1526 2023-12-06                1.380746   
1126     1126  um_242-20231109T1526 2023-12-07                1.391720   
1127     1127  um_242-20231109T1526 2023-12-08                1.362308   

     aragonite_status  index_y    lon_max    lon_min    lat_max    lat_min  \
0           Saturated        8 -7

In [9]:
#now do the same with vessel data
#1. Open Dataset
dsv=xr.open_dataset(root / raw_data / "Vessels" / "vessel_based_bottom_OA_data_2007_2024.nc")
dfv = dsv.to_dataframe()
print(dfv)

#2. Reset your index and check columns
dfv=dfv.reset_index()
print("Original Columns:", dfv.columns.tolist())

#3. Classify aragonite
dfvc = classify_arag(dfv, 'omega_bottom')
dfvd=calculate_daily_averages(dfvc, "cruise", "time", "omega_bottom")


#4. classify your aragonite state from the daily averages
dfvdc=classify_arag(dfvd, 'omega_bottom_daily_avg')

#5. Get extent
dfve = get_extent(dfv, 'time', 'lon', 'lat', 'depth')
dfve=dfve.reset_index()

#6. Combine with your spatial bounds (min/max lat and lon) according to the time column
df_final = combine_df(dfvdc, dfve, 'time')
print(df_final)

#7. Export it 
export_path = root / "data" / "Processed_Data" / "vessel_based_bottom_OA_extent_average_2007_2024.csv"
#df_final.to_csv(export_path, date_format='%Y-%m-%d %H:%M:%S', index=False)



               data_source   cruise obs_type accession      lat      lon  \
time                                                                       
2007-07-23  CODAP_NA_v2021  GOMECC1   Niskin     83633  35.9640 -74.9535   
2007-07-23  CODAP_NA_v2021  GOMECC1   Niskin     83633  35.9162 -74.8502   
2007-07-24  CODAP_NA_v2021  GOMECC1   Niskin     83633  35.8982 -74.7913   
2007-07-24  CODAP_NA_v2021  GOMECC1   Niskin     83633  35.8323 -74.6477   
2007-07-24  CODAP_NA_v2021  GOMECC1   Niskin     83633  35.6043 -74.2607   
...                    ...      ...      ...       ...      ...      ...   
2024-11-07      NCEI-OCADS   PC2406   Niskin   0309698  36.0000 -74.7800   
2024-11-08      NCEI-OCADS   PC2406   Niskin   0309698  37.9980 -74.9633   
2024-11-08      NCEI-OCADS   PC2406   Niskin   0309698  37.8470 -74.5800   
2024-11-08      NCEI-OCADS   PC2406   Niskin   0309698  37.7070 -74.2617   
2024-11-10      NCEI-OCADS   PC2406   Niskin   0309698  39.0600 -72.7300   

           

In [ ]:
ds=xr.open_dataset(root / raw_data / "Vessels" / "vessel_based_bottom_OA_data_2007_2024.nc")
df=ds.to_dataframe()
df

Classify omega_arag values and drop values >= the drop threshold (1.4).
- Returns data with lat, lon, depth, pH at each point.

In [ ]:
#Full process with vessel dataset
dsv = xr.open_dataset(root / "data" / "Raw_Data" / "Vessels" / "vessel_based_bottom_OA_data_2007_2024.nc")
dfv=dsv.to_dataframe()
#print(dfv)
dfv=dfv.reset_index()
print(dfv.columns.to_list())
vessel_extent = get_extent(dfv, "cruise", "lon", "lat", 'depth')
#vessel_extent.to_excel(root / "data" / "Processed_Data" / "vessel_based_bottom_OA_extent_2007_2024.xlsx", index=False)
dfvd=calculate_daily_averages(dfv, 'time', 'cruise', 'omega_bottom')
dfvd=classify_arag(dfvd, 'omega_bottom_daily_avg')
dfv_final=combine_df(dfvd, vessel_extent, 'cruise')
print(dfv_final)
# Step 4: Export it 
export_path = root / "data" / "Processed_Data" / "vessel_based_bottom_OA_extent_average_2007_2024.csv"
dfv_final.to_csv(export_path, date_format='%Y-%m-%d %H:%M:%S', index=False)


['time', 'data_source', 'cruise', 'obs_type', 'accession', 'lat', 'lon', 'depth', 'bottom_depth', 'pH_bottom', 'omega_bottom', 'temperature_bottom']
     cruise        date  omega_bottom_daily_avg  \
0    DE1202  2012-02-08                1.136667   
1    DE1202  2012-02-09                1.300000   
2    DE1202  2012-02-10                1.395000   
3    DE1202  2012-02-11                1.236667   
4     ECOA1  2015-06-21                1.386667   
..      ...         ...                     ...   
109  PC2305  2023-10-28                1.310000   
110  PC2305  2023-11-09                1.110000   
111  PC2305  2023-11-10                1.200000   
112  PC2305  2023-11-11                1.070000   
113  PC2305  2023-11-13                1.380000   

                             aragonite_status  lon_max  lon_min  lat_max  \
0            Atlantic Cod threshold surpassed -65.7733 -75.5333  43.8550   
1                                  suboptimal -65.7733 -75.5333  43.8550   
2         